# 数据处理与预处理工具

本教程介绍 PipelineTS 的数据加载、预处理和工具函数：

1. **内置数据集**: 加载各种时序数据集
2. **数据生成器**: 生成合成时序数据
3. **数据缩放器 (Scaler)**: 数据标准化与反标准化
4. **序列分割**: 将时序数据分割为监督学习格式
5. **评估指标**: 内置各种预测评估指标
6. **区间预测准确率**: quantile_acc

## 1. 内置数据集

PipelineTS 提供多个内置数据集，适合快速实验。

In [ ]:
from PipelineTS.dataset import (
    LoadElectricDataSets,
    LoadMessagesSentHourDataSets,
    LoadMessagesSentDataSets,
    LoadWebSales,
    LoadSupermarketIncoming,
    BuiltInSeriesData
)

datasets = {
    'Electric': LoadElectricDataSets,
    'MessagesSentHour': LoadMessagesSentHourDataSets,
    'MessagesSent': LoadMessagesSentDataSets,
    'WebSales': LoadWebSales,
    'SupermarketIncoming': LoadSupermarketIncoming,
}

for name, loader in datasets.items():
    df = loader()
    print(f"{name}: shape={df.shape}, columns={df.columns.tolist()}")

## 2. 数据生成器

使用 `DataGenerator` 生成合成时序数据，适合测试和基准测试。

In [ ]:
from PipelineTS.dataset import DataGenerator

# 生成合成时间序列
synthetic_data = DataGenerator(n=200)
print(type(synthetic_data))
synthetic_data

## 3. 数据缩放器 (Scaler)

PipelineTS 提供统一的 `Scaler` 接口，支持 4 种缩放方式。

In [ ]:
import numpy as np
from PipelineTS.preprocessing import Scaler

# 准备数据
X = np.random.randn(100, 1)

# 支持的缩放器类型: 'min_max', 'standard', 'quantile', 'gauss_rank'
for scaler_name in ['min_max', 'standard', 'quantile', 'gauss_rank']:
    scaler = Scaler(scaler_name)
    transformed = scaler.fit_transform(X)
    recovered = scaler.inverse_transform(transformed)
    print(f"{scaler_name:12s} | 缩放后范围: [{transformed.min():.3f}, {transformed.max():.3f}] | "
          f"恢复误差: {np.abs(X - recovered).max():.2e}")

## 4. 序列分割

将一维/多维时间序列转换为监督学习格式 (X, y)。

In [ ]:
from PipelineTS.spinesTS.preprocessing import (
    split_series,
    train_test_split_ts,
    lag_splits,
    split_series_multivariate
)

# 单变量序列分割
series = np.sin(np.linspace(0, 4 * np.pi, 100))
X, y = split_series(series, in_features=10, out_features=5)
print(f"split_series: X.shape={X.shape}, y.shape={y.shape}")

# 时序训练/测试集分割（保持时序顺序）
X_train, X_test, y_train, y_test = train_test_split_ts(X, y, train_size=0.8)
print(f"train: X={X_train.shape}, y={y_train.shape}")
print(f"test:  X={X_test.shape}, y={y_test.shape}")

In [ ]:
# 多变量序列分割（3D 数据）
multi_series = np.random.randn(100, 3).astype(np.float32)
X_mv, y_mv = split_series_multivariate(multi_series, in_features=10, out_features=5)
print(f"split_series_multivariate: X.shape={X_mv.shape}, y.shape={y_mv.shape}")
print(f"  维度含义: (样本数, 时间步, 变量数)")

## 5. 评估指标

In [ ]:
from PipelineTS.spinesTS.metrics import mae, mse, rmse, wmape

y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = np.array([1.1, 2.2, 2.8, 4.1, 5.3])

print(f"MAE:   {mae(y_true, y_pred):.4f}")
print(f"MSE:   {mse(y_true, y_pred):.4f}")
print(f"RMSE:  {rmse(y_true, y_pred):.4f}")
print(f"WMAPE: {wmape(y_true, y_pred):.4f}")

## 6. 区间预测准确率 (quantile_acc)

评估预测区间的覆盖率。

In [ ]:
from PipelineTS.metrics import quantile_acc

y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
lower  = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
upper  = np.array([1.5, 2.5, 3.5, 4.5, 5.5])

acc = quantile_acc(y_true, lower, upper)
print(f"区间覆盖率: {acc:.2%}")

## 总结

| 功能 | API |
|------|-----|
| 内置数据集 | `LoadElectricDataSets()`, `LoadWebSales()` 等 |
| 数据生成 | `DataGenerator(n=100)` |
| 数据缩放 | `Scaler('min_max')` / `Scaler('standard')` |
| 序列分割 | `split_series()`, `split_series_multivariate()` |
| 评估指标 | `mae()`, `rmse()`, `wmape()` |
| 区间准确率 | `quantile_acc(yt, lower, upper)` |